# Loan Fairness Analysis

### Task logic (short)
- Train a classifier to predict `repaid` **without using `group`** or `applicant_id`.
- Evaluate the model separately for groups A and B using **precision, recall, and approval rate**.
- If the results are very different, the model is treating the groups differently even though it never received `group`.
- Train a second classifier using the same neutral features to predict `group`. A high AUC shows that the supposedly neutral features contain information that can act as a proxy for group.
- Try **group-specific threshold tuning**. Here, the B threshold is lowered until the recall gap is minimized.
- Report which fairness measure improved and which one became worse.

**Fairness idea:** threshold tuning can improve one fairness measure (here, recall parity) while hurting another (here, precision parity). There is usually a trade-off between fairness measures.

In [1]:
# ============================================================
# LOAN FAIRNESS ANALYSIS - COMPLETE TASK IN ONE CELL
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report
)

# -----------------------------
# 1. Load the dataset
# -----------------------------
df = pd.read_csv("loan_applications.csv")

print("Dataset shape:", df.shape)
print("\nColumns:", list(df.columns))

# group = protected/group variable used ONLY for fairness evaluation
# applicant_id = identifier, so it should not be used as a feature
target = "repaid"
group_col = "group"

feature_cols = [
    "income",
    "years_employed",
    "credit_score",
    "debt_ratio",
    "loan_amount"
]

X = df[feature_cols].copy()
y = df[target].copy()
groups = df[group_col].copy()

# -----------------------------
# 2. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X,
    y,
    groups,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# -----------------------------
# 3. Train classifier WITHOUT group
# -----------------------------
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

# Probability of predicted positive class (repaid = 1)
prob_test = model.predict_proba(X_test)[:, 1]

# Normal threshold = 0.50
pred_test = (prob_test >= 0.50).astype(int)

# -----------------------------
# 4. Fairness metrics by group
# -----------------------------
def calculate_group_metrics(y_true, y_pred, group_values):
    results = {}

    for group in ["A", "B"]:
        mask = np.asarray(group_values) == group

        results[group] = {
            "precision": precision_score(
                np.asarray(y_true)[mask],
                np.asarray(y_pred)[mask],
                zero_division=0
            ),
            "recall": recall_score(
                np.asarray(y_true)[mask],
                np.asarray(y_pred)[mask],
                zero_division=0
            ),
            "approval_rate": np.mean(np.asarray(y_pred)[mask])
        }

    return pd.DataFrame(results).T


baseline = calculate_group_metrics(y_test, pred_test, g_test)

print("\n" + "=" * 65)
print("1. BASELINE FAIRNESS RESULTS (threshold = 0.50)")
print("=" * 65)
print(baseline.round(4))

print("\nGaps (absolute difference between A and B):")
print("Precision gap:", round(abs(baseline.loc["A", "precision"] -
                                  baseline.loc["B", "precision"]), 4))
print("Recall gap:", round(abs(baseline.loc["A", "recall"] -
                               baseline.loc["B", "recall"]), 4))
print("Approval-rate gap:", round(abs(baseline.loc["A", "approval_rate"] -
                                      baseline.loc["B", "approval_rate"]), 4))

# -----------------------------
# 5. Find out how the model can infer group
#    from neutral features
# -----------------------------
# IMPORTANT:
# This second model is ONLY for investigation.
# It asks: "Can the neutral features predict group?"
group_target = (groups == "A").astype(int)

Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X,
    group_target,
    test_size=0.30,
    stratify=group_target,
    random_state=42
)

group_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

group_model.fit(Xg_train, yg_train)
group_prob = group_model.predict_proba(Xg_test)[:, 1]

group_auc = roc_auc_score(yg_test, group_prob)

print("\n" + "=" * 65)
print("2. HOW CAN GROUP BE INFERRED FROM 'NEUTRAL' FEATURES?")
print("=" * 65)
print("AUC of neutral-feature -> group classifier:", round(group_auc, 4))

coef = group_model.named_steps["classifier"].coef_[0]
feature_importance = (
    pd.DataFrame({
        "feature": feature_cols,
        "coefficient": coef,
        "absolute_coefficient": np.abs(coef)
    })
    .sort_values("absolute_coefficient", ascending=False)
)

print("\nFeatures most useful for inferring group:")
print(feature_importance[["feature", "coefficient"]].round(4).to_string(index=False))

# -----------------------------
# 6. Threshold tuning
# -----------------------------
# We keep A at 0.50 and search for a B threshold that
# makes A and B recall as close as possible.
best_threshold_B = None
best_recall_gap = float("inf")
best_tuned = None

for threshold_B in np.arange(0.05, 0.96, 0.01):
    tuned_pred = np.where(
        np.asarray(g_test) == "A",
        prob_test >= 0.50,
        prob_test >= threshold_B
    ).astype(int)

    tuned_metrics = calculate_group_metrics(
        y_test,
        tuned_pred,
        g_test
    )

    recall_gap = abs(
        tuned_metrics.loc["A", "recall"] -
        tuned_metrics.loc["B", "recall"]
    )

    if recall_gap < best_recall_gap:
        best_recall_gap = recall_gap
        best_threshold_B = round(float(threshold_B), 2)
        best_tuned = tuned_metrics.copy()

print("\n" + "=" * 65)
print("3. THRESHOLD TUNING")
print("=" * 65)
print("Threshold for Group A:", 0.50)
print("Tuned threshold for Group B:", best_threshold_B)

print("\nAfter threshold tuning:")
print(best_tuned.round(4))

print("\nNew gaps:")
print("Precision gap:", round(abs(best_tuned.loc["A", "precision"] -
                                  best_tuned.loc["B", "precision"]), 4))
print("Recall gap:", round(abs(best_tuned.loc["A", "recall"] -
                               best_tuned.loc["B", "recall"]), 4))
print("Approval-rate gap:", round(abs(best_tuned.loc["A", "approval_rate"] -
                                      best_tuned.loc["B", "approval_rate"]), 4))

# -----------------------------
# 7. Automatically report what improved/broke
# -----------------------------
baseline_gaps = {
    "precision": abs(baseline.loc["A", "precision"] -
                     baseline.loc["B", "precision"]),
    "recall": abs(baseline.loc["A", "recall"] -
                  baseline.loc["B", "recall"]),
    "approval_rate": abs(baseline.loc["A", "approval_rate"] -
                         baseline.loc["B", "approval_rate"])
}

tuned_gaps = {
    "precision": abs(best_tuned.loc["A", "precision"] -
                     best_tuned.loc["B", "precision"]),
    "recall": abs(best_tuned.loc["A", "recall"] -
                  best_tuned.loc["B", "recall"]),
    "approval_rate": abs(best_tuned.loc["A", "approval_rate"] -
                         best_tuned.loc["B", "approval_rate"])
}

print("\n" + "=" * 65)
print("4. FAIRNESS CONCLUSION")
print("=" * 65)

for metric in ["precision", "recall", "approval_rate"]:
    change = tuned_gaps[metric] - baseline_gaps[metric]

    if change < 0:
        status = "IMPROVED (smaller A/B gap)"
    elif change > 0:
        status = "WORSENED (larger A/B gap)"
    else:
        status = "UNCHANGED"

    print(
        f"{metric:15s}: "
        f"baseline gap={baseline_gaps[metric]:.4f}, "
        f"tuned gap={tuned_gaps[metric]:.4f} -> {status}"
    )

print("\nPlain-English interpretation:")
print(
    "The model never receives the group column, but the other features "
    "contain enough information to act as proxies for group membership."
)
print(
    f"The neutral-feature group classifier has AUC={group_auc:.3f}; "
    "an AUC well above 0.50 means group membership is predictable."
)
print(
    f"Lowering Group B's threshold to {best_threshold_B:.2f} was used to "
    "reduce the recall gap. This demonstrates that threshold tuning can "
    "improve one fairness measure while worsening another."
)


Dataset shape: (6000, 8)

Columns: ['applicant_id', 'group', 'income', 'years_employed', 'credit_score', 'debt_ratio', 'loan_amount', 'repaid']

1. BASELINE FAIRNESS RESULTS (threshold = 0.50)
   precision  recall  approval_rate
A     0.6381  0.1816          0.081
B     0.6667  0.0211          0.006

Gaps (absolute difference between A and B):
Precision gap: 0.0286
Recall gap: 0.1605
Approval-rate gap: 0.075

2. HOW CAN GROUP BE INFERRED FROM 'NEUTRAL' FEATURES?
AUC of neutral-feature -> group classifier: 0.8317

Features most useful for inferring group:
       feature  coefficient
        income       1.2027
  credit_score       0.7142
years_employed       0.4086
   loan_amount      -0.0774
    debt_ratio       0.0732

3. THRESHOLD TUNING
Threshold for Group A: 0.5
Tuned threshold for Group B: 0.36

After threshold tuning:
   precision  recall  approval_rate
A     0.6381  0.1816         0.0810
B     0.5152  0.1789         0.0656

New gaps:
Precision gap: 0.1229
Recall gap: 0.0026
Appr